# 01 噪声源分类与建模框架（1 月）

目标：系统梳理芯片端与测控端噪声来源，形成可落地的仿真参数映射。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('未找到文件 pyproject.toml 和 examples/backend.yaml?')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.ui.notebook import run_workflow

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'examples' / 'noise_simulation_tests' / 'runs' / 'roadmap_2026H1'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)


def run_case(tag: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, engine: str = 'qutip'):
    out_dir = OUT_ROOT / tag
    return run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        persist_artifacts=True,
        export_dxf=False,
    )


def get_metric(result: dict, key: str, default: float = np.nan) -> float:
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    return float(obs.get(key, default))

In [ ]:
QASM_1Q = """
OPENQASM 3;
qubit[1] q;
bit[1] c;
h q[0];
rz(0.5) q[0];
x q[0];
measure q[0] -> c[0];
"""

In [ ]:
noise_framework = {
    '???': ['T1??', 'T2???', '???(gamma_up)', '????'],
    '???': ['AWG????(control_scale)', '??????(gate_duration)', '??????(OU/1f??)'],
    '???': ['??(couplings??????)', '???????'],
}

for k, v in noise_framework.items():
    print(f'[{k}]')
    for item in v:
        print(' -', item)